# Follow an indexed output back to its source

An index can reorder values and pick the same source element more than once. This notebook follows one output position at a time, first with a static figure and then with keyboard controls. Static figures also work in hosts that cannot display widgets.


In [ ]:
import numpy as np
from IPython.display import display

import rainbow_tensor as rt

x = np.arange(12).reshape(3, 4)
selection = ([2, 0, 2], slice(None, None, -2))
display(x)
display(rt.index(x, selection, show_result=True))


## Read the result in order

The row index chooses rows 2, 0 and 2. The slice `::-2` starts at the last column and moves backwards by two, choosing columns 3 and 1. The first and last output rows repeat because both came from source row 2.


In [ ]:
result = x[selection]
assert result.tolist() == [[11, 9], [3, 1], [11, 9]]
result


## Follow the first output

`focus` is a coordinate in the result. Output `(0, 0)` is the first value, 11. Its source coordinate is `(2, 3)`. Passing `focus` automatically shows the source and result side by side.


In [ ]:
focused = rt.index(x, selection, focus=(0, 0))
first_source = focused.trace.terms[0][0].coordinate
assert first_source == (2, 3)
assert focused.trace.output_coord == (0, 0)
display(focused)
print('Output (0, 0) reads source', first_source)


## Two outputs can share one source

Output `(2, 0)` is also 11. It is a separate output position even though it reads the same source element. The trace describes the focused output. The index mapping can answer any valid output coordinate without changing focus.


In [ ]:
repeated = rt.index(x, selection, focus=(2, 0))
assert repeated.trace.terms[0][0].coordinate == first_source
assert focused.index_mapping.source_coord((0, 0)) == (2, 3)
assert focused.index_mapping.source_coord((2, 0)) == (2, 3)
display(repeated)


## Try the keyboard controls

The next cell opens an explorer at output `(0, 0)`. Type `2` in the first coordinate field and `0` in the second, then press **Update focus**. The output highlight moves while the source coordinate stays `(2, 3)`.

Install `rainbow-tensor` in the notebook kernel's environment. Live controls need a running kernel and widget support in its host. For hosts without widget support, use `display(focused)` to show the static figure instead.


In [ ]:
index_explorer = rt.explore(rt.index, x, selection)
assert index_explorer.focus == (0, 0)
display(index_explorer)


## Change focus from Python

Output `(1, 1)` is the second value in the middle output row. It comes from source `(0, 1)` and has value 1. The next cell updates the controls when they are available, or creates the same static figure otherwise.


In [ ]:
index_explorer.set_focus((1, 1))
current_visual = index_explorer.visual
source_coordinate = current_visual.trace.terms[0][0].coordinate
assert source_coordinate == (0, 1)
assert x[source_coordinate] == 1
display(current_visual)


## A scalar output uses `()`

Selecting source row 2 and column 3 removes both axes. The result is a scalar with one addressable output coordinate, `()`. Its source coordinate still has two components. A scalar explorer has no coordinate fields because there are no output axes to edit.


In [ ]:
scalar_visual = rt.index(x, (2, 3), focus=())
assert scalar_visual.trace.output_coord == ()
assert scalar_visual.trace.terms[0][0].coordinate == (2, 3)
display(scalar_visual)


## An empty output has no focus

The slice `0:0` selects no rows. There is no output coordinate to follow, so the trace is `None`. With widgets installed, the empty explorer has no coordinate fields and its update button is disabled. Trying to set a focus would raise `IndexError`.


In [ ]:
empty_selection = (slice(0, 0), slice(None))
empty_explorer = rt.explore(rt.index, x, empty_selection)
assert empty_explorer.focus is None
empty_visual = empty_explorer.visual
display(empty_explorer)
assert empty_visual.trace is None
assert empty_visual.index_mapping.result_count == 0


## Keep a figure when you need it

`current_visual` is an ordinary static result. To export it, run the following yourself with a path you choose. This notebook does not write an export automatically.

```python
current_visual.save('focused-index.svg')
```

The saved SVG does not need a running notebook kernel. The live controls do. When you have finished exploring, release the controls with:

```python
if index_explorer is not None:
    index_explorer.close()
if empty_explorer is not None:
    empty_explorer.close()
```

Without `focus`, `rt.index(x, selection)` continues to show the usual static source selection. Use it when the whole selection matters more than one output position.
